In [ ]:
# roc_curve = sklearn.metrics.roc_curve(y_train_df['Coarse Label'], val_preds_prob_knn)

In [1]:


# # Create feature matrix for test data using same vocab_dict
# V = len(vocab_list)
# rows_test = []
# for text in x_test_df['text'].values:
#     vec = np.zeros(V, dtype=int)
#     for tok in tokenize_text(text):
#         if tok in vocab_dict:
#             vec[vocab_dict[tok]] += 1
#     rows_test.append(vec)
# X_test_counts = np.vstack(rows_test)

# # Add vocab columns to test DataFrame
# vocab_test_df = pd.DataFrame(X_test_counts, columns=vocab_cols, index=x_test_df.index)
# x_test_aug = pd.concat([x_test_df, vocab_test_df], axis=1)

# print("Test DataFrame shape:", x_test_aug.shape)
# print(x_test_aug.head())

In [2]:
# def build_vocabulary(tokenized_text_list, min, max):
#   tok_count_dict = dict()

#   for tokenized_text in tokenized_text_list:
#     for token in tokenized_text:
#       if token not in tok_count_dict:
#         tok_count_dict[token] = 1 # Initialize count for new token
#       else:
#         tok_count_dict[token] += 1 # Increment count for existing token
  
#   # Sort the tokens by their counts in descending order
#   sorted_tokens = list(sorted(tok_count_dict, key=tok_count_dict.get, reverse=True))

#   # # Print out the 10 most common tokens and their counts
#   # for w in sorted_tokens[:10]:
#   #   print("%5d %s" % (tok_count_dict[w], w))
  
#   # # Print out the 10 least common tokens and their counts
#   # for w in sorted_tokens[-10:]:
#   #   print("%5d %s" % (tok_count_dict[w], w))

#   conjunctions = [
#       # Coordinating conjunctions
#       "nor", "yet",

#       # Subordinating conjunctions
#       "after", "although", "as", "because", "before", "once", "since", "though", "unless",
#       "until", "whenever", "whereas", "wherever", "whether", "while",

#       # Correlative conjunctions
#       "either", "neither"
#   ]
  
#   # Merge plural counts into singular (e.g., 'dogs' -> 'dog'), skip empty tokens
#   for t in list(tok_count_dict.keys()):
#     if not t:
#       continue
#     if len(t) > 1 and t.endswith('s') and t[:-1] in tok_count_dict:
#       tok_count_dict[t[:-1]] += tok_count_dict[t]
#       tok_count_dict[t] = 0

#   vocab_list = [w for w in sorted_tokens if ((tok_count_dict[w] >= min and tok_count_dict[w] <= max and len(w) >= 6) or w in conjunctions)]
  
#   return vocab_list, tok_count_dict

In [3]:
# training_text = x_train_df['text'].values.tolist()
# tokenized_training_text = [tokenize_text(text) for text in training_text]
# print("Example tokenized texts:")
# for i, tokens in enumerate(tokenized_training_text[:5]):
#     print(f"Text {i}: {tokens}")

In [4]:
# def tokenize_text(raw_text):
#     ''' Transform a plain-text string into a list of tokens
    
#     We assume that *whitespace* divides tokens.
    
#     Args
#     ----
#     raw_text : string
    
#     Returns
#     -------
#     list_of_tokens : list of strings
#         Each element is one token in the provided text
#     '''
#     list_of_tokens = raw_text.split() # split method divides on whitespace by default
#     for pp in range(len(list_of_tokens)):
#         cur_token = list_of_tokens[pp]
#         # Remove punctuation
#         for punc in ['?', '!', '_', '.', ',', '"', '/']:
#             cur_token = cur_token.replace(punc, "")
#         # Turn to lower case
#         clean_token = cur_token.lower()
#         # Replace the cleaned token into the original list
#         list_of_tokens[pp] = clean_token
#     return list_of_tokens

In [5]:
import numpy as np
import pandas as pd
import os
import textwrap

In [6]:
def load_data(data_dir, x_filename='x_train.csv', y_filename='y_train.csv'):
    x_train_df = pd.read_csv(os.path.join(data_dir, x_filename))
    y_train_df = pd.read_csv(os.path.join(data_dir, y_filename))

    N, n_cols = x_train_df.shape
    print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
    print("Shape of y_train_df: %s" % str(y_train_df.shape))

    # Print out 8 random entries
    tr_text_list = x_train_df['text'].values.tolist()
    prng = np.random.RandomState(101)
    rows = prng.permutation(np.arange(y_train_df.shape[0]))
    for row_id in rows[:3]:
        text = tr_text_list[row_id]
        print("row %5d | %s BY %s | y = %s" % (
            row_id,
            y_train_df['title'].values[row_id],
            y_train_df['author'].values[row_id],
            y_train_df['Coarse Label'].values[row_id],
            ))
        # Pretty print text via textwrap library
        line_list = textwrap.wrap(tr_text_list[row_id],
            width=70,
            initial_indent='  ',
            subsequent_indent='  ')
        print('\n'.join(line_list))
        print("")
    return x_train_df, y_train_df

In [7]:
x_train_df, y_train_df = load_data(data_dir = "data", x_filename = "x_train.csv", y_filename = "y_train.csv")

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)
row  4746 | The Red and the Black: A Chronicle of 1830 BY Stendhal | y = Key Stage 4-5
  It was hermetically sealed; he was on the point of  fainting and
  remained for a long time leaning against the oak; then  with a
  staggering step he went to have another look at the gardener's
  ladder. The chain which he had once forced asunder--in, alas, such
  different  circumstances--had not yet been repaired. Carried away by
  a moment of  madness, Julien pressed it to his lips.

row  1250 | Cranford BY Elizabeth Cleghorn Gaskell | y = Key Stage 4-5
  Miss Pole, Miss Matty, and I, meanwhile attended to Miss Brown: and
  hard  work we found it to relieve her querulous and never-ending
  complaints. But if we were so weary and dispirited, what must Miss
  Jessie have been! Yet she came back almost calm as if she had gained
  a new strength. She  put off her mourning dress, and came in,
  looking pale and gentle,  thanking us each 

In [8]:
import sklearn
from sklearn.feature_extraction.text import TfidfTransformer

def make_logit_pipeline(C=1.0, use_tfidf=True, sublinear_tf=True):
    """Construct a sklearn Pipeline that optionally applies TF-IDF to count features, then scales and fits a logistic regression."""
    steps = []
    if use_tfidf:
        steps.append(('tfidf', TfidfTransformer(sublinear_tf=sublinear_tf)))
    steps.extend([
         #('rescaler', sklearn.preprocessing.MinMaxScaler()),
         ('logit', sklearn.linear_model.LogisticRegression(solver="lbfgs", l1_ratio=0, C=C, max_iter=1000))
    ])
    pipeline = sklearn.pipeline.Pipeline(steps=steps)
    return pipeline

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

def find_best_vocab(
    x_df=x_train_df, 
    y_labels=y_train_df['Coarse Label'], 
    min_df_range=(10, 51, 5), 
    max_df_range=np.arange(1.0, 0.4, -.1), 
    n_splits=10, 
    random_state=42
):
    hypers_list = []
    kf = sklearn.model_selection.KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for min_df in min_df_range:
        for max_df in max_df_range:
            # Build feature matrix for this vocab
            # Use CountVectorizer to build count matrix; TF-IDF will be applied in the pipeline
            vectorizer = CountVectorizer(min_df=min_df, max_df=max_df, binary=True)
            X_train = vectorizer.fit_transform(x_df['text'])
            analyze = vectorizer.build_analyzer()
            vocab = vectorizer.get_feature_names_out()
            X_counts = X_train.toarray()
            vocab_df = pd.DataFrame(X_counts, columns=vocab, index=x_df.index)
            
            # CV evaluation with this vocab
            pl = make_logit_pipeline(C=1.0)
            aucs = []
            for train_index, val_index in kf.split(vocab_df):
                x_train_fold = vocab_df.iloc[train_index]
                y_train_fold = y_labels.iloc[train_index]
                x_val_fold = vocab_df.iloc[val_index]
                y_val_fold = y_labels.iloc[val_index]
                
                pl.fit(x_train_fold, y_train_fold)
                y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]
                auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
                aucs.append(auc)
            
            mean_auc = np.mean(aucs)
            hypers_list.append((min_df, max_df, len(vocab), mean_auc))
    
    # Get best vocab bounds
    best_hypers_config = max(hypers_list, key=lambda x: x[3])
    best_min_df, best_max_df, best_vocab_length, best_auc = best_hypers_config
    
    return {
        'best_min_df': best_min_df,
        'best_max_df': best_max_df,
        'best_vocab_length': best_vocab_length,
        'best_auc': best_auc
    }

In [10]:
# Run tuning
results = find_best_vocab()
best_min_df = results['best_min_df']
best_max_df = results['best_max_df']

print(f"\n✓ Best bounds: min={best_min_df}, max={best_max_df}")
print(f"  Vocab size: {results['best_vocab_length']}")
print(f"  Best AUC: {results['best_auc']:.4f}")


✓ Best bounds: min=5, max=1.0
  Vocab size: 6456
  Best AUC: 0.8029


In [11]:
# Load and augment test data with same vocabulary
x_test_df = pd.read_csv(os.path.join("data", "x_test.csv"))

vectorizer = CountVectorizer(min_df=best_min_df, max_df=best_max_df, binary=True)
X_train = vectorizer.fit_transform(x_train_df['text'])
analyze = vectorizer.build_analyzer()
vocab = vectorizer.get_feature_names_out()
print(vocab[:5])
X_train_counts = X_train.toarray()
print("X train counts:")
print(X_train_counts[:5])
X_test = vectorizer.transform(x_test_df['text'])
X_test_counts = X_test.toarray()
print("X test counts: ")
print(X_test_counts[:5])

x_train_df = pd.DataFrame(X_train_counts, columns=vocab, index=x_train_df.index)
x_test_df = pd.DataFrame(X_test_counts, columns=vocab, index=x_test_df.index)

['000' '10' '100' '11' '12']
X train counts:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
X test counts: 
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [12]:
best_pl = make_logit_pipeline(C=1.0)
best_pl.fit(x_train_df, y_train_df['Coarse Label'])
y_test_pred_proba = best_pl.predict_proba(x_test_df)[:, 1]
with open("yproba1_test.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")

In [16]:
C_grid = np.logspace(-4, 4, 17)
hypers_list = []

kf = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
for C in C_grid:
  pl = make_logit_pipeline(C=C)
  aucs = []
  for train_index, val_index in kf.split(x_train_df):
    x_train_fold = x_train_df.iloc[train_index]
    y_train_fold = y_train_df['Coarse Label'].iloc[train_index]
    x_val_fold = x_train_df.iloc[val_index]
    y_val_fold = y_train_df['Coarse Label'].iloc[val_index]

    pl.fit(x_train_fold, y_train_fold)
    y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]

    auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
    aucs.append(auc)
  mean_auc = np.mean(aucs)
  hypers_list.append((C, mean_auc))

best_C, best_auc = max(hypers_list, key=lambda x: x[1])
print(f"Best C: {best_C}, Best AUC: {best_auc}")

Best C: 10.0, Best AUC: 0.8145228968337355


In [18]:
best_pl = make_logit_pipeline(C=1.0)
best_pl.fit(x_train_df, y_train_df['Coarse Label'])
y_test_pred_proba = best_pl.predict_proba(x_test_df)[:, 1]
with open("yproba1_test.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")